<a href="https://colab.research.google.com/github/wetherc/data-2000/blob/sp26/homework/050_random-forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WORK IN PROGRESS

# Lab Assignment: Predicting Storm Fatalities and Property Damage with Ensemble Models

## Overview

In this lab you will work with a large, multi-file dataset of weather events recorded across the United States by the National Oceanic and Atmospheric Administration (NOAA). Your goal is to build, tune, and compare a **Random Forest** and an **XGBoost model** — first as classifiers predicting whether a storm event caused any fatalities, then as regressors predicting the dollar value of property damage. This dataset is substantially messier than anything you have worked with so far. It arrives as multiple separate files that must be assembled, contains numeric values embedded in strings, uses inconsistent geographic coding, and has extreme class imbalance on the target variable. Navigating these issues carefully is the central skill this assignment develops.

## The Dataset

The NOAA Storm Events Database is maintained by the National Centers for Environmental Information (NCEI) and contains records of weather events in the United States dating back to 1950. Events are entered by trained meteorologists at local National Weather Service offices and are drawn from a combination of automated sensors, storm surveys, emergency management reports, and news sources. Because data entry is performed by humans across dozens of regional offices, the database reflects genuine real-world inconsistencies in how the same type of information gets recorded in different places at different times.

For this assignment you will work with **three to five years of your choosing** from the period 2015–2023. Each year of data is distributed as three separate CSV files:

- **`StormEvents_details`** — One row per event, containing event type, location, timing, narrative descriptions, and damage estimates. This is your primary file.
- **`StormEvents_fatalities`** — One row per fatality associated with an event, containing information about the victim. This must be aggregated to the event level and joined to the details file.
- **`StormEvents_locations`** — One row per affected location within an event (some events span multiple counties or cities). Useful for geographic feature engineering but optional for the core modeling tasks.

All files are available for download at:
**https://www.ncdc.noaa.gov/stormevents/ftp.jsp**

Files are named with the pattern `StormEvents_details-ftp_v1.0_dYYYY_cYYYYMMDD.csv.gz`. They are gzip-compressed and can be read directly by pandas without manual decompression using `pd.read_csv("filename.csv.gz")`.

After assembling and joining your files, your working dataset should contain somewhere between **150,000 and 250,000 event records** depending on the years selected.

### Features

The details file contains roughly 50 columns. What follows describes the most important ones and the modeling considerations each raises.

#### Identification and Timing

**`EVENT_ID`** — A unique integer identifier for each event. Use this as your join key when merging the fatalities file. Do not use it as a predictor.

**`YEAR`**, **`MONTH_NAME`**, **`BEGIN_DAY`**, **`BEGIN_TIME`** — Temporal information for the event's start. `BEGIN_TIME` is stored as a four-digit integer in HHMM format (e.g., 1430 means 2:30 PM). You will need to engineer useful time features from these columns. Consider extracting hour of day, season (from month), and whether the event occurred at night (roughly 8 PM to 6 AM) — darkness is associated with higher fatality rates because people are less aware of developing hazards.

**`END_DAY`**, **`END_TIME`** — Event end time, same format. Engineer an **`DURATION_HOURS`** feature representing how long the event lasted. Be cautious of events that span midnight (end time numerically less than begin time) and events that span multiple days.

#### Geography

**`STATE`** — Full state name, uppercase. Useful as a categorical feature but has 50+ levels including territories. Consider grouping by census region (Northeast, Midwest, South, West) to reduce cardinality.

**`CZ_TYPE`** — Whether the event was recorded for a county (`C`) or a forecast zone (`Z`). Forecast zones do not correspond to county boundaries and are used primarily for marine and some mountain events. This distinction matters if you attempt any spatial joins.

**`CZ_NAME`** — Name of the county or zone. Not directly useful as a model feature but helpful for debugging geographic anomalies.

**`BEGIN_LAT`**, **`BEGIN_LON`** — Latitude and longitude of the event start point. These are missing for a substantial fraction of records, particularly older ones and zone-based events. Do not drop records solely on this basis; instead treat missingness as informative and create a boolean **`HAS_COORDS`** flag.

#### Event Characteristics

**`EVENT_TYPE`** — The type of weather event. This is one of your most important features. There are 68 recognized event types in the current taxonomy, ranging from `Tornado` and `Hurricane` to `Dust Devil` and `Astronomical Low Tide`. Event types vary enormously in their typical damage, duration, and fatality risk. You should retain this as a categorical feature. Be aware that the taxonomy changed over time — before 2012 many event types were collapsed into broader categories, so if you include pre-2012 data you will encounter type labels not present in later years.

**`SOURCE`** — How the event was reported: trained spotter, emergency manager, law enforcement, broadcast media, and so on. This is a proxy for data quality and geographic coverage. Encode as categorical.

**`MAGNITUDE`** and **`MAGNITUDE_TYPE`** — Numeric magnitude of the event (e.g., wind speed in knots, hail size in inches) and the measurement type. Many event types have no magnitude (e.g., floods, winter storms), so this column is missing for the majority of records. When present it is highly informative. Engineer a **`HAS_MAGNITUDE`** boolean and use the magnitude value itself with missing values filled to 0 or imputed within event type.

**`TOR_F_SCALE`** — For tornadoes, the Enhanced Fujita scale rating: EF0 through EF5. Missing for all non-tornado events, and occasionally missing even for tornado events when the rating could not be determined (recorded as `EFU` for unknown). Map the known ratings to integers 0–5, treat `EFU` and missing as a separate category or impute conservatively.

**`TOR_LENGTH`** and **`TOR_WIDTH`** — Tornado path length in miles and width in yards. Continuous, missing for non-tornado events. These are among the strongest predictors of tornado damage and fatalities when present.

#### Damage

**`DAMAGE_PROPERTY`** and **`DAMAGE_CROPS`** — Estimated dollar value of property and crop damage. These are the most important and most problematic columns in the dataset. They are stored as **strings with suffix multipliers**: values like `"2.5K"`, `"300K"`, `"1.2M"`, and `"5B"` representing thousands, millions, and billions of dollars respectively. You must write a parsing function to convert these to numeric values before any modeling. A zero value is stored as `"0"` or sometimes left blank.

This parsing function is one of the most important pieces of code you will write in this assignment. It must handle at minimum: the suffixes K, M, B (case-insensitive); numeric strings with no suffix; empty strings and `NaN`; and the occasional malformed entry. Test it explicitly on edge cases before applying it.

After parsing, `DAMAGE_PROPERTY` is your regression target. Its distribution is extremely right-skewed — the vast majority of events cause zero or near-zero damage, while a small number of major hurricanes and tornadoes account for the bulk of total dollar losses. A **log transformation** (`log1p`, which handles zeros gracefully) is strongly recommended.

**`DEATHS_DIRECT`** and **`DEATHS_INDIRECT`** — Direct deaths (caused immediately by the storm) and indirect deaths (caused by conditions created by the storm, such as a car accident during a blizzard). These columns exist in the details file as totals. Your classification target is a binary indicator: **did this event cause any fatalities?** You will engineer this as `(DEATHS_DIRECT + DEATHS_INDIRECT) > 0`. Note that you can also derive this from the fatalities join file and use the two as a cross-check.

**`INJURIES_DIRECT`** and **`INJURIES_INDIRECT`** — Injury counts, same structure as deaths. These are predictors, not targets. Be thoughtful about leakage: injuries and fatalities are correlated outcomes of the same event, so including injury counts as predictors of fatality occurrence creates a form of data leakage if both are recorded simultaneously. Discuss this issue in your write-up.

#### Narrative Text

**`EVENT_NARRATIVE`** and **`EPISODE_NARRATIVE`** — Free-text descriptions of the event written by NWS meteorologists. These are rich sources of information but require NLP techniques to use directly. For this assignment you should engineer at least two simple text-derived features: **`NARRATIVE_LENGTH`** (character count of the event narrative, a proxy for event severity and documentation quality) and a boolean flag for whether the narrative mentions a keyword associated with severe outcomes (e.g., whether the word "fatality", "death", "destroyed", or "swept" appears). You are not required to build a full text feature pipeline, but you should engage with the text columns in some way.

### The Fatalities Join

The fatalities file (`StormEvents_fatalities`) contains one row per individual fatality with columns including `EVENT_ID`, `FATALITY_TYPE` (Direct or Indirect), `FATALITY_AGE`, `FATALITY_SEX`, and `FATALITY_LOCATION` (where the person was when killed: in a vehicle, indoors, outdoors, etc.).

To use this file you must aggregate it to the event level and join it to the details file. At minimum, compute the total fatality count per `EVENT_ID`. If you want richer features, you can also compute the fraction of fatalities that were direct, the average victim age, or the most common fatality location per event.

After joining, events with no matching rows in the fatalities file will have `NaN` fatality counts. These are **true zeros** — no fatalities occurred — not missing data. Fill them with 0 before modeling.

### Known Data Quality Issues

This dataset has more data quality challenges than any other dataset in this course sequence. The following issues are the most consequential, but careful inspection will reveal others.

- **The damage string format.** As described above, `DAMAGE_PROPERTY` and `DAMAGE_CROPS` are not numeric. Writing a robust parser is mandatory and non-trivial. Do not attempt to coerce them to numeric directly; pandas will silently produce `NaN` for every non-numeric string.

- **Inconsistent geographic encoding.** State names are uppercase full names in some columns and two-letter abbreviations in others. FIPS codes appear in some columns but not others. Do not assume that geographic columns can be joined across files without cleaning.

- **Temporal edge cases.** Events that span midnight, events with missing end times, and events with implausibly long durations (some records have clerical errors placing the end date years after the begin date) all require handling. Cap or remove duration outliers rather than letting them distort your feature distributions.

- **Class imbalance.** Fatal storm events are rare. Depending on your year selection, somewhere between 2% and 5% of events in the dataset resulted in any fatality. This means that a naive classifier that predicts "no fatality" for every event will achieve 95%+ accuracy while being completely useless. You must address this explicitly — through class weighting (`class_weight="balanced"` in sklearn), oversampling (SMOTE), undersampling, or adjusting the classification threshold — and you must report metrics beyond accuracy (AUC-ROC and F1 at minimum).

- **Duplicate or near-duplicate records.** Some events are entered multiple times with slightly different begin times or geographic codes. A brief deduplication check on `EVENT_ID` is worthwhile; true duplicate IDs should be dropped.

- **Missing `BEGIN_LAT` / `BEGIN_LON`.** Roughly 40% of records lack coordinate information. Do not drop these records. Create the `HAS_COORDS` flag described above and treat missingness as a feature.

- **Zero-damage majority class.** For the regression task, the majority of events have exactly zero property damage. Consider whether you want to model all events together, or whether a two-stage model (first predict whether damage occurred, then predict the amount conditional on damage occurring) would be more appropriate. Discuss the tradeoffs even if you implement only the single-stage version.

## Modeling Concepts

**Class imbalance** is the central modeling challenge in the classification task. When one class is rare, most standard algorithms will bias toward the majority class because doing so minimizes overall loss. The two main remedies available in sklearn and XGBoost are: (1) setting `class_weight="balanced"` or `scale_pos_weight` (XGBoost), which adjusts the loss function to penalize minority class errors more heavily; and (2) resampling the training data to artificially balance the classes. You should try at least one of these approaches and compare results to an unweighted baseline.

**Log transformation of the regression target** is necessary when the target spans many orders of magnitude, as `DAMAGE_PROPERTY` does. When you evaluate your regression model, report error metrics both in log-space (where you trained) and in the original dollar scale (where the numbers are interpretable). Use `np.expm1` to invert the `log1p` transformation. Be aware that errors in dollar space will be dominated by the largest events, which is often appropriate — being wrong by \$10M on a hurricane is worse than being wrong by \$10M on a hailstorm.

**Feature leakage** deserves explicit attention here because the injury and fatality columns are both outcomes of the same event. If you include `INJURIES_DIRECT` as a predictor of fatality occurrence, you are using information that would only be known after the event is over — the same moment at which you would know whether fatalities occurred. Whether this constitutes leakage depends on the intended use case: if you are building a post-event damage assessment tool, injuries are a legitimate input; if you are building a real-time early warning system, they are not. State your assumed use case clearly and let it guide your feature selection.

## Your Tasks

### Part 1: Data Assembly and Cleaning

1. Download three to five years of Storm Events data. Load, concatenate, and join the details and fatalities files as described above. Report row and column counts at each step.

2. Write and test a `parse_damage()` function that converts the string damage format to a float. Apply it to both `DAMAGE_PROPERTY` and `DAMAGE_CROPS`. Show before-and-after value counts for a sample of records to confirm it worked correctly.

3. Engineer the following features: `DURATION_HOURS`, `IS_NIGHT` (boolean), season from `MONTH_NAME`, `HAS_MAGNITUDE`, `HAS_COORDS`, `NARRATIVE_LENGTH`, and a keyword-presence flag from `EVENT_NARRATIVE`. Document any edge cases you encountered while building each one.

4. Create your two target variables: `FATAL_EVENT` (binary: any fatalities) and `LOG_DAMAGE` (`log1p` of `DAMAGE_PROPERTY`). Report the class balance for `FATAL_EVENT` and the distribution of `LOG_DAMAGE`.

5. Address remaining missing values. Identify which columns have meaningful missingness and which have structural missingness, and handle each appropriately.

### Part 2: Classification — Did This Event Kill Anyone?

6. Train a baseline Random Forest classifier with default hyperparameters and no class weighting. Report accuracy, precision, recall, F1, and AUC-ROC on the test set. Comment on what the accuracy score obscures.

7. Retrain with `class_weight="balanced"`. Report the same metrics. Describe what changed and why — specifically, what did the model trade away in order to improve recall on the minority class?

8. Train and tune an XGBoost classifier. Use `scale_pos_weight` to address class imbalance, setting it to the ratio of negative to positive examples in the training set. Tune at least three additional hyperparameters. Report the same metrics.

9. Plot ROC curves for both tuned models on the same axes. Which model achieves higher AUC? At an operating point where recall on fatal events is at least 0.70, what is the precision of each model?

### Part 3: Regression — How Much Property Damage Did This Event Cause?

10. Train a baseline Random Forest regressor on `LOG_DAMAGE`. Report RMSE and R² in log-space on the test set.

11. Tune the Random Forest over at least three hyperparameters. Report improvement over baseline.

12. Train and tune an XGBoost regressor. Report RMSE and R² in log-space.

13. For both tuned models, convert predictions back to dollar scale using `np.expm1` and compute RMSE in dollars. Plot predicted vs. actual dollar damage on a log-log scale. Where do the models perform well, and where do they break down?

### Part 4: Interpretation

14. Plot feature importances for all four tuned models. Which features are consistently most important? Are there features that matter for damage prediction but not for fatality prediction, or vice versa?

15. `EVENT_TYPE` is almost certainly near the top of your importance rankings. Using partial dependence or simply grouping your test set by event type, describe which event types the model finds easiest and hardest to predict. Offer a substantive explanation.

16. Revisit the leakage question from the modeling concepts section. Does your final model include injury counts as features? Justify your choice in two to three sentences with reference to your assumed use case.


## Assembly Instructions

Because this dataset arrives as multiple files across multiple years, assembly is itself a significant step. The following pseudocode outlines the recommended approach:

```python
import pandas as pd
import glob

# 1. Load and concatenate all details files
detail_files = glob.glob("data/StormEvents_details*.csv.gz")
details = pd.concat([pd.read_csv(f, low_memory=False) for f in detail_files], ignore_index=True)

# 2. Load and concatenate all fatalities files
fatality_files = glob.glob("data/StormEvents_fatalities*.csv.gz")
fatalities = pd.concat([pd.read_csv(f, low_memory=False) for f in fatality_files], ignore_index=True)

# 3. Aggregate fatalities to event level
fat_agg = fatalities.groupby("EVENT_ID").size().reset_index(name="FATALITY_COUNT")

# 4. Left join to details (preserves all events, including those with no fatalities)
df = details.merge(fat_agg, on="EVENT_ID", how="left")
df["FATALITY_COUNT"] = df["FATALITY_COUNT"].fillna(0).astype(int)
```

Verify your row counts at each step. After the join you should have the same number of rows as the details file.